<a href="https://colab.research.google.com/github/kauefs/ML/blob/%40/notebooks/AnomalyDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

<font color=F0F0F0 size=8px font-family=Georgia>
<strong><ins>ƊⱭȾɅViƧi&#x1F9FF;Ƞ</ins>&trade;</strong></font>

[![ƊⱭȾɅViƧi🧿Ƞ](https://img.shields.io/badge/ƊⱭȾɅViƧi&#x1F9FF;Ƞ&trade;-0065FF?style=plastic)](https://datavision.one/)

[![GitHub     ](https://img.shields.io/badge/-000000?logo=github&logoColor=FFFFFF)](https://github.com/kauefs/)
[![Medium     ](https://img.shields.io/badge/-000000?logo=medium&logoColor=FFFFFF)](https://medium.com/@kauefs)
[![LinkedIn   ](https://img.shields.io/badge/-2867B2?logo=linkedin&logoColor=FFFFFF)](https://www.linkedin.com/in/kauefs/)
[![Python     ](https://img.shields.io/badge/3-646464?logo=python&logoColor=FFDE57&labelColor=4584B6)](https://www.python.org/)

[![License]( https://img.shields.io/badge/Apache--2.0-D22128?style=flat&logo=apache&logoColor=CB2138&label=License&labelColor=6D6E71&color=D22128)](https://www.apache.org/licenses/LICENSE-2.0)

![2024.12.20  ](https://img.shields.io/badge/2024.12.20-000000)

In [1]:
!date

Sat Dec 21 05:33:53 AM UTC 2024


---
<center>
    <a heref=https://pycaret.readthedocs.io/en/latest/index.html><img src=https://pycaret.org/wp-content/uploads/2020/03/Divi93_43.png></a>
</center>

# Anomaly Detection

In [2]:
# @title Libraries
%pip install -U -q pip pandas matplotlib mlflow umap-learn pycaret
import      numpy            as   np
import     pandas            as   pd
import matplotlib.pyplot     as   plt
import    seaborn            as   sns
import                            sys
import     mlflow
import       umap
from       mlflow          import    *
from         umap          import    *
from      pycaret.datasets import get_data
from      pycaret.anomaly  import setup, create_model, assign_model, plot_model, predict_model, save_model, load_model, evaluate_model, AnomalyExperiment
from      pycaret.anomaly  import AnomalyExperiment, models, get_config, set_config, evaluate_model, save_experiment, load_experiment
# Configurations:
print('System Ready: Python –',   sys.version)
%matplotlib inline
%config     InlineBackend.figure_format='svg'
# %config     InlineBackend.print_figure_kwargs={'bbox_inches':None}
# %precision 2
pd.options.plotting.matplotlib.register_converters=True
pd.set_option('display.max.columns',               None)
plt.rcParams[      'font.family']      =                                        'sans-serif'
sns.set_theme(context='notebook', style='whitegrid', palette='colorblind', font='sans-serif', font_scale=1.15, color_codes=True, rc={'grid.color':'1','grid.linestyle':':'})

System Ready: Python – 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]


In [3]:
# @title Data
# LoadData:
data=get_data('anomaly')

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10
0,0.263995,0.764929,0.138424,0.935242,0.605867,0.518790,0.912225,0.608234,0.723782,0.733591
1,0.546092,0.653975,0.065575,0.227772,0.845269,0.837066,0.272379,0.331679,0.429297,0.367422
2,0.336714,0.538842,0.192801,0.553563,0.074515,0.332993,0.365792,0.861309,0.899017,0.088600
3,0.092108,0.995017,0.014465,0.176371,0.241530,0.514724,0.562208,0.158963,0.073715,0.208463
4,0.325261,0.805968,0.957033,0.331665,0.307923,0.355315,0.501899,0.558449,0.885169,0.182754


In [4]:
# Data Info:
data.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Col1    1000 non-null   float64
 1   Col2    1000 non-null   float64
 2   Col3    1000 non-null   float64
 3   Col4    1000 non-null   float64
 4   Col5    1000 non-null   float64
 5   Col6    1000 non-null   float64
 6   Col7    1000 non-null   float64
 7   Col8    1000 non-null   float64
 8   Col9    1000 non-null   float64
 9   Col10   1000 non-null   float64
dtypes: float64(10)
memory usage: 78.2 KB


In [5]:
# @title Train & Test Split
test  =data.sample(frac=.10)
train =data.drop(test.index)
test.reset_index( inplace=True, drop=True)
train.reset_index(inplace=True, drop=True)
print('\t',train.shape,
    '\n\t', test.shape)

	 (900, 10) 
	 (100, 10)


In [6]:
# @title SetUp
# SetUp Help:
help(setup)

Help on function setup in module pycaret.anomaly.functional:

setup(data: Union[dict, list, tuple, numpy.ndarray, scipy.sparse._matrix.spmatrix, pandas.core.frame.DataFrame, NoneType] = None, data_func: Optional[Callable[[], Union[dict, list, tuple, numpy.ndarray, scipy.sparse._matrix.spmatrix, pandas.core.frame.DataFrame]]] = None, index: Union[bool, int, str, list, tuple, numpy.ndarray, pandas.core.series.Series] = True, ordinal_features: Optional[Dict[str, list]] = None, numeric_features: Optional[List[str]] = None, categorical_features: Optional[List[str]] = None, date_features: Optional[List[str]] = None, text_features: Optional[List[str]] = None, ignore_features: Optional[List[str]] = None, keep_features: Optional[List[str]] = None, preprocess: bool = True, create_date_columns: List[str] = ['day', 'month', 'year'], imputation_type: Optional[str] = 'simple', numeric_imputation: str = 'mean', categorical_imputation: str = 'mode', text_features_method: str = 'tf-idf', max_encoding_o

In [7]:
# SetUp:
anomaly=setup(data=train, normalize=True, session_id=0, experiment_name='anomaly')

,Description,Value
0,Session id,0
1,Original data shape,"(900, 10)"
2,Transformed data shape,"(900, 10)"
3,Numeric features,10
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Normalize,True
9,Normalize method,zscore


In [8]:
# Available Config:
get_config()

{'USI',
 'X',
 'X_train',
 'X_train_transformed',
 'X_transformed',
 '_available_plots',
 '_ml_usecase',
 'data',
 'dataset',
 'dataset_transformed',
 'exp_id',
 'exp_name_log',
 'gpu_n_jobs_param',
 'gpu_param',
 'html_param',
 'idx',
 'is_multiclass',
 'log_plots_param',
 'logging_param',
 'memory',
 'n_jobs_param',
 'pipeline',
 'seed',
 'train',
 'train_transformed',
 'variable_and_property_keys',
 'variables'}

In [9]:
# Accessing    Seed:
print('Current Seed: {}'.format(get_config('seed')))
# Changing     Seed:
# set_config(   'seed'    ,                   123    )
# print('New     Seed: {}'.format(get_config('seed')))

Current Seed: 0


In [10]:
# @title Create Model
# Verifying Models:
models()

,Name,Reference
ID,,
abod,Angle-base Outlier Detection,pyod.models.abod.ABOD
cluster,Clustering-Based Local Outlier,pycaret.internal.patches.pyod.CBLOFForceToDouble
cof,Connectivity-Based Local Outlier,pyod.models.cof.COF
iforest,Isolation Forest,pyod.models.iforest.IForest
histogram,Histogram-based Outlier Detection,pyod.models.hbos.HBOS
knn,K-Nearest Neighbors Detector,pyod.models.knn.KNN
lof,Local Outlier Factor,pyod.models.lof.LOF
svm,One-class SVM detector,pyod.models.ocsvm.OCSVM
pca,Principal Component Analysis,pyod.models.pca.PCA


In [11]:
# Create Model Help:
help(create_model)

Help on function create_model in module pycaret.anomaly.functional:

create_model(model: Union[str, Any], fraction: float = 0.05, verbose: bool = True, fit_kwargs: Optional[dict] = None, experiment_custom_tags: Optional[Dict[str, Any]] = None, **kwargs)
    This function trains a given model from the model library. All available
    models can be accessed using the ``models`` function.
    
    
    Example
    -------
    >>> from pycaret.datasets import get_data
    >>> anomaly = get_data('anomaly')
    >>> from pycaret.anomaly import *
    >>> exp_name = setup(data = anomaly)
    >>> knn = create_model('knn')
    
    
    model: str or scikit-learn compatible object
        ID of an model available in the model library or pass an untrained
        model object consistent with scikit-learn API. Estimators available
        in the model library (ID - Name):
    
        * 'abod' - Angle-base Outlier Detection
        * 'cluster' - Clustering-Based Local Outlier
        * 'cof' - Conn

In [12]:
# Create Model:
pca=create_model('pca')
# Verifying ParaMeters:
pca

Processing:   0%|          | 0/3 [00:00<?, ?it/s]

PCA(contamination=0.05, copy=True, iterated_power='auto', n_components=None,
  n_selected_components=None, random_state=0, standardization=True,
  svd_solver='auto', tol=0.0, weighted=True, whiten=False)

In [13]:
# Type:
type(pca)

pyod.models.pca.PCA

In [14]:
# @title Assign Model
# Assign Model Help
help(assign_model)

Help on function assign_model in module pycaret.anomaly.functional:

assign_model(model, transformation: bool = False, score: bool = True, verbose: bool = True) -> pandas.core.frame.DataFrame
    This function assigns anomaly labels to the dataset for a given model.
    (1 = outlier, 0 = inlier).
    
    
    Example
    -------
    >>> from pycaret.datasets import get_data
    >>> anomaly = get_data('anomaly')
    >>> from pycaret.anomaly import *
    >>> exp_name = setup(data = anomaly)
    >>> knn = create_model('knn')
    >>> knn_df = assign_model(knn)
    
    
    model: scikit-learn compatible object
        Trained model object
    
    
    transformation: bool, default = False
        Whether to apply anomaly labels on the transformed dataset.
    
    
    score: bool, default = True
        Whether to show outlier score or not.
    
    
    verbose: bool, default = True
        Status update is not printed when verbose is set to False.
    
    
    Returns:
        panda

In [15]:
# Model Instance
# Model Designation (Assign) & Recording Results to a Variable:
results=assign_model(pca)
# Label/Anomaly: 0 = No Anomaly & 1 = With Anomaly
# Anomaly Score:        Anomaly Score
results#.head()

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,0.263995,0.764929,0.138424,0.935242,0.605867,0.518790,0.912225,0.608234,0.723782,0.733591,0,805.943919
1,0.546092,0.653975,0.065575,0.227772,0.845269,0.837066,0.272379,0.331679,0.429297,0.367422,0,683.175699
2,0.336714,0.538842,0.192801,0.553563,0.074515,0.332993,0.365792,0.861309,0.899017,0.088600,0,902.459358
3,0.325261,0.805968,0.957033,0.331665,0.307923,0.355315,0.501899,0.558449,0.885169,0.182754,0,799.043911
4,0.212465,0.780305,0.458444,0.634509,0.373030,0.465651,0.413997,0.013080,0.570250,0.736672,0,698.881862
...,...,...,...,...,...,...,...,...,...,...,...,...
895,0.305055,0.656837,0.331665,0.822525,0.907127,0.882276,0.855732,0.584786,0.808640,0.242762,0,773.302071
896,0.812627,0.864258,0.616604,0.167966,0.811223,0.938071,0.418462,0.472306,0.348347,0.671129,0,698.729545
897,0.250967,0.138627,0.919703,0.461234,0.886555,0.869888,0.800908,0.530324,0.779433,0.234952,0,817.069359
898,0.502436,0.936820,0.580062,0.540773,0.151995,0.059452,0.225220,0.242755,0.279385,0.538755,0,753.534388


In [16]:
# Verifying Results for Label == 1:
results[results['Anomaly'] == 1]

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
6,0.869237,0.277979,0.423076,0.112472,0.183727,0.034960,0.111114,0.249330,0.550683,0.049843,1,966.384669
10,0.796623,0.230543,0.993018,0.077075,0.094068,0.718628,0.977611,0.333386,0.634843,0.028729,1,1017.055388
19,0.162825,0.674069,0.705447,0.774799,0.894267,0.443057,0.399779,0.009136,0.941851,0.982711,1,988.439964
22,0.057884,0.227162,0.022494,0.167069,0.631315,0.610103,0.277753,0.728503,0.716414,0.973370,1,975.459812
27,0.834861,0.973891,0.808398,0.222828,0.018934,0.708188,0.986392,0.545479,0.235898,0.844788,1,957.999811
28,0.669766,0.694779,0.191048,0.080716,0.441480,0.115675,0.054466,0.949300,0.318667,0.410017,1,942.131001
32,0.249776,0.276445,0.905772,0.843307,0.622943,0.757426,0.656207,0.887785,0.040510,0.808190,1,935.870685
40,0.178173,0.304310,0.028749,0.800844,0.161153,0.874650,0.984053,0.205747,0.029727,0.842088,1,1034.894163
49,0.273081,0.097663,0.491502,0.065955,0.087729,0.727527,0.004372,0.903040,0.254568,0.402991,1,950.802912
55,0.981709,0.253236,0.695971,0.918359,0.108317,0.656997,0.806218,0.924069,0.916568,0.406505,1,978.091164


In [17]:
# @title Plot Model
# Checking DocString for Available Plots:
help(plot_model)

Help on function plot_model in module pycaret.anomaly.functional:

plot_model(model, plot: str = 'tsne', feature: Optional[str] = None, label: bool = False, scale: float = 1, save: bool = False, display_format: Optional[str] = None) -> Optional[str]
    This function analyzes the performance of a trained model.
    
    
    Example
    -------
    >>> from pycaret.datasets import get_data
    >>> anomaly = get_data('anomaly')
    >>> from pycaret.anomaly import *
    >>> exp_name = setup(data = anomaly)
    >>> knn = create_model('knn')
    >>> plot_model(knn, plot = 'tsne')
    
    
    model: scikit-learn compatible object
        Trained Model Object
    
    
    plot: str, default = 'tsne'
        List of available plots (ID - Name):
    
        * 'tsne' - t-SNE (3d) Dimension Plot
        * 'umap' - UMAP Dimensionality Plot
    
    
    feature: str, default = None
        Feature to be used as a hoverover tooltip and/or label when the ``label``
        param is set to True. 

In [18]:
# Visualyzing/Analysing Results – tSNE:
plot_model(pca         ,    plot='tsne')

In [19]:
# Visualyzing/Analysing Results – UMAP:
plot_model(pca         ,    plot='umap')

In [20]:
# # Test Predictions:
predictions=predict_model(pca, data=test)
# Verifying Predictions:
predictions#.head()

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,-1.710996,0.055250,1.581712,-0.224471,0.953706,0.291418,-0.042769,0.649157,-0.523387,0.979981,0,706.495324
1,-1.556816,1.981025,-1.905968,-1.210341,-1.038838,-0.000825,0.189737,-1.362684,-1.668756,-1.352670,1,1047.519400
2,-1.696922,0.186878,-0.593644,-0.359845,-1.434838,-1.279352,1.268013,0.911062,1.190367,-0.751265,0,815.716003
3,-1.799183,-0.067348,1.737263,-0.212815,-1.454541,-1.372023,1.197457,0.743684,1.195752,-0.875281,0,898.604086
4,0.903328,0.209710,1.835348,-0.039473,0.654920,0.131069,1.525368,0.882318,1.566223,-1.199386,0,813.023609
...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.829905,-0.233314,0.478071,-1.265075,1.064893,1.215546,1.436378,0.719061,1.543411,-1.288400,0,817.368536
96,-1.629686,-0.920098,0.672940,-0.126304,0.839491,1.005736,1.035478,0.497486,1.122700,-0.965519,0,735.556402
97,-0.517968,0.452519,0.548592,0.013100,-1.615810,-1.252932,-0.955745,-1.622076,-0.336469,-0.797227,0,733.207104
98,0.760734,0.451004,1.641725,-0.175392,0.491224,1.205472,-0.730244,-1.566600,-0.057038,-1.076261,0,722.421767


In [21]:
# @title Save/Load Model
# Saving  Model:
save_model(pca,  'PyCaretModelPCA')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['Col1', 'Col2', 'Col3', 'Col4',
                                              'Col5', 'Col6', 'Col7', 'Col8',
                                              'Col9', 'Col10'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=[],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('normalize', TransformerWrapper(transformer=StandardScaler())),
                 ('trained_model',
                  PCA(contamination=0.05, copy=True, iterated_power='auto', n_components=None,
   n_selected_components=None, random_state=0, standardization=True,
   svd_solver='auto', tol=0.0, weighted=True, whiten=False))]),
 'PyCaretModelPCA.pkl')

In [22]:
# Loading Model:
model=load_model('PyCaretModelPCA')

Transformation Pipeline and Model Successfully Loaded


In [23]:
# New Predictions:
new=predict_model(model, data=test)
# Verifying Results:
new#.head()

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,0.052495,0.506366,0.907679,0.438323,0.905472,0.607198,0.497363,0.602278,0.351128,0.703629,0,706.495309
1,0.092108,0.995017,0.014465,0.176371,0.241530,0.514724,0.562208,0.158963,0.073715,0.208463,1,1047.519380
2,0.056111,0.539765,0.350559,0.402354,0.109578,0.110161,0.862937,0.659989,0.766205,0.336127,0,815.716038
3,0.029838,0.475257,0.947517,0.441421,0.103012,0.080837,0.843259,0.623107,0.767509,0.309801,0,898.604050
4,0.724176,0.545559,0.972637,0.487479,0.805912,0.556459,0.934712,0.653656,0.857239,0.241002,0,813.023591
...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.705312,0.433145,0.625031,0.161827,0.942521,0.899618,0.909893,0.617682,0.851713,0.222106,0,817.368527
96,0.073386,0.258878,0.674938,0.464407,0.867414,0.833228,0.798083,0.568857,0.749816,0.290646,0,735.556377
97,0.359012,0.607170,0.643091,0.501448,0.049276,0.118521,0.242736,0.101806,0.396400,0.326370,0,733.207091
98,0.687541,0.606785,0.923049,0.451364,0.751367,0.896430,0.305628,0.114030,0.464079,0.267138,0,722.421775


In [24]:
# @title Experiment
experiment=AnomalyExperiment()
# Type:
type(experiment)

pycaret.anomaly.oop.AnomalyExperiment

In [25]:
# Experiment SetUp:
experiment.setup(data=train, normalize=True, session_id=0, experiment_name='anomaly')

,Description,Value
0,Session id,0
1,Original data shape,"(900, 10)"
2,Transformed data shape,"(900, 10)"
3,Numeric features,10
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Normalize,True
9,Normalize method,zscore


In [26]:
# Model Creation/Training:
iforest=experiment.create_model('iforest')
iforest

Processing:   0%|          | 0/3 [00:00<?, ?it/s]

IForest(behaviour='new', bootstrap=False, contamination=0.05,
    max_features=1.0, max_samples='auto', n_estimators=100, n_jobs=-1,
    random_state=0, verbose=0)

In [27]:
# Model Assignment:
# Assigns Anomaly Labels to Taining Data
anomalies=experiment.assign_model(iforest)
anomalies

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,0.263995,0.764929,0.138424,0.935242,0.605867,0.518790,0.912225,0.608234,0.723782,0.733591,0,-0.015787
1,0.546092,0.653975,0.065575,0.227772,0.845269,0.837066,0.272379,0.331679,0.429297,0.367422,0,-0.080828
2,0.336714,0.538842,0.192801,0.553563,0.074515,0.332993,0.365792,0.861309,0.899017,0.088600,1,0.004413
3,0.325261,0.805968,0.957033,0.331665,0.307923,0.355315,0.501899,0.558449,0.885169,0.182754,0,-0.015964
4,0.212465,0.780305,0.458444,0.634509,0.373030,0.465651,0.413997,0.013080,0.570250,0.736672,0,-0.032098
...,...,...,...,...,...,...,...,...,...,...,...,...
895,0.305055,0.656837,0.331665,0.822525,0.907127,0.882276,0.855732,0.584786,0.808640,0.242762,0,-0.085529
896,0.812627,0.864258,0.616604,0.167966,0.811223,0.938071,0.418462,0.472306,0.348347,0.671129,0,-0.078626
897,0.250967,0.138627,0.919703,0.461234,0.886555,0.869888,0.800908,0.530324,0.779433,0.234952,0,-0.067786
898,0.502436,0.936820,0.580062,0.540773,0.151995,0.059452,0.225220,0.242755,0.279385,0.538755,0,-0.075605


In [28]:
# Plot Model
# Model Analysis/Visualyzing Results – tSNE:
plot_model(iforest          ,    plot='tsne')

In [29]:
# Plot Model
# Model Analysis/Visualyzing Results – UMAP:
plot_model(iforest          ,    plot='umap')

In [30]:
# Model Evaluation:
experiment.evaluate_model(iforest)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [31]:
# Test Prediction:
predictions=experiment.predict_model(iforest, data=test)
predictions

,Col1,Col2,Col3,Col4,Col5,Col6,Col7,Col8,Col9,Col10,Anomaly,Anomaly_Score
0,-0.887793,1.074247,-1.421955,1.645702,0.054566,0.012025,1.444739,0.676189,1.015213,1.121128,0,-0.015787
1,0.210187,0.636977,-1.706402,-1.016890,0.773034,1.017864,-0.849460,-0.578871,-0.200645,-0.603839,0,-0.080828
2,-0.604757,0.183241,-1.209630,0.209237,-1.540066,-0.575143,-0.514522,1.824688,1.738714,-1.917326,1,0.004413
3,-1.556816,1.981025,-1.905968,-1.210341,-1.038838,-0.000825,0.189737,-1.362684,-1.668756,-1.352670,1,0.055171
4,-0.649334,1.235982,1.774422,-0.625885,-0.839587,-0.504600,-0.026506,0.450256,1.681542,-1.473780,0,-0.015964
...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.727980,0.648259,-0.667418,1.221488,0.958674,1.160740,1.242179,0.569774,1.365572,-1.191093,0,-0.085529
996,1.247596,1.465704,0.445166,-1.241973,0.670859,1.337067,-0.325673,0.059323,-0.534867,0.826879,0,-0.078626
997,-0.938502,-1.394009,1.628662,-0.138245,0.896935,1.121590,1.045608,0.322618,1.244981,-1.227886,0,-0.067786
998,0.040266,1.751670,0.302483,0.161102,-1.307540,-1.439605,-1.018549,-0.982420,-0.819596,0.203285,0,-0.075605


In [32]:
# @title Save/Load Experiment
# Save Experiment:
save_experiment(           'ExperimentPipeLine')
# Load Experiment:
experiment=load_experiment('ExperimentPipeLine', data=data)

,Description,Value
0,Session id,0
1,Original data shape,"(1000, 10)"
2,Transformed data shape,"(1000, 10)"
3,Numeric features,10
4,Preprocess,True
5,Imputation type,simple
6,Numeric imputation,mean
7,Categorical imputation,mode
8,Normalize,True
9,Normalize method,zscore
